# Annual TorNet manifest generation

Build, validate, and persist one or more annual TorNet raw manifests.

The initial run is intentionally limited to 2014. Raw artifacts are preserved with
an explicit `_INVALID.json` marker when required validation failures are found.
After this workflow is validated, it will be extended to 2015–2022.


In [ ]:
%pip install -q xarray netCDF4 pandas pyarrow


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
%pip install --force-reinstall --no-deps "/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.1-py3-none-any.whl"


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

import tornado_detection

from tornado_detection.data.manifest import (
    EXPECTED_CATEGORIES,
    EXPECTED_SPLITS,
    build_archive_manifests,
    validate_manifests,
    write_manifest_artifacts,
)

print(
    "tornado_detection package version:",
    tornado_detection.__version__,
)
print(
    "Loaded tornado_detection from:",
    tornado_detection.__file__,
)


## Configuration


In [ ]:
TORNET_ARCHIVE_DIR = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)

MANIFEST_OUTPUT_ROOT = (
    TORNET_ARCHIVE_DIR
    / "manifests"
    / "v1"
)

MANIFEST_WORK_DIR = Path(
    "/content/tornet_manifest_work"
)

YEARS_TO_BUILD = (2014,)

EXPECTED_COMBINATIONS = {
    (split, category)
    for split in EXPECTED_SPLITS
    for category in EXPECTED_CATEGORIES
}

EXPECTED_DIMENSIONS = {
    "time": 4,
    "sweep": 2,
    "azimuth": 120,
    "range": 240,
    "lims": 2,
}

print("Years to build:", YEARS_TO_BUILD)
print("Archive directory:", TORNET_ARCHIVE_DIR)
print("Output root:", MANIFEST_OUTPUT_ROOT)


## Build and validate one year


In [ ]:
def build_and_write_year(year: int):
    archive_path = (
        TORNET_ARCHIVE_DIR
        / f"tornet_{year}.tar.gz"
    )

    output_directory = (
        MANIFEST_OUTPUT_ROOT
        / str(year)
    )

    if not archive_path.is_file():
        raise FileNotFoundError(archive_path)

    existing_terminal_markers = [
        path
        for path in [
            output_directory / "_SUCCESS.json",
            output_directory / "_INVALID.json",
        ]
        if path.exists()
    ]

    if existing_terminal_markers:
        raise FileExistsError(
            "A completed annual audit already exists: "
            + ", ".join(
                str(path)
                for path in existing_terminal_markers
            )
        )

    print()
    print("=" * 72)
    print(f"Building TorNet raw audit for {year}")
    print("Archive:", archive_path)
    print("Output:", output_directory)
    print("=" * 72)

    result = build_archive_manifests(
        archive_path,
        expected_year=year,
        working_directory=MANIFEST_WORK_DIR,
        progress_every=1_000,
        progress=print,
    )

    actual_combinations = {
        (row.split, row.category)
        for row in (
            result.file_manifest[
                ["split", "category"]
            ]
            .drop_duplicates()
            .itertuples(index=False)
        )
    }

    if actual_combinations != EXPECTED_COMBINATIONS:
        raise AssertionError(
            f"Unexpected split/category combinations for {year}: "
            f"actual={sorted(actual_combinations)}, "
            f"expected={sorted(EXPECTED_COMBINATIONS)}"
        )

    if (
        len(result.file_manifest)
        != result.netcdf_member_count
    ):
        raise AssertionError(
            "File-manifest count does not match scanned "
            f"NetCDF members for {year}: "
            f"files={len(result.file_manifest):,}, "
            f"members={result.netcdf_member_count:,}"
        )

    validation = validate_manifests(
        result,
        expected_file_count=(
            result.netcdf_member_count
        ),
        expected_frame_count=(
            result.netcdf_member_count * 4
        ),
        expected_frames_per_file=4,
        expected_dimensions=EXPECTED_DIMENSIONS,
    )

    display(validation.checks)
    display(
        validation.category_frame_summary
    )
    display(result.schema_summary)

    print(
        "Event groups crossing official splits:"
    )
    display(validation.event_split_overlap)

    if not validation.event_split_overlap.empty:
        overlap_event_ids = set(
            validation.event_split_overlap[
                "event_group_id"
            ].astype(str)
        )

        overlap_files = (
            result.file_manifest.loc[
                result.file_manifest[
                    "event_group_id"
                ].astype(str).isin(
                    overlap_event_ids
                )
            ]
            .sort_values(
                [
                    "event_group_id",
                    "split",
                    "archive_member",
                ]
            )
            .reset_index(drop=True)
        )

        overlap_columns = [
            "archive_member",
            "split",
            "category",
            "event_group_id",
            "episode_id",
            "radar_site",
            "frame_time_start_utc",
            "frame_time_end_utc",
            "positive_frame_count",
            "frame_labels_json",
            "member_sha256",
        ]

        print(
            "Files participating in official-split "
            "event overlaps:"
        )
        display(
            overlap_files[overlap_columns]
        )

    print(
        "Episode groups crossing official splits "
        "(informational):"
    )
    display(validation.episode_split_overlap)

    artifacts = write_manifest_artifacts(
        result,
        validation,
        output_directory,
        overwrite=False,
        allow_invalid=True,
    )

    status = (
        "valid"
        if validation.all_required_passed
        else "invalid"
    )
    marker_name = (
        "_SUCCESS.json"
        if status == "valid"
        else "_INVALID.json"
    )

    summary = {
        "year": year,
        "status": status,
        "netcdf_members": (
            result.netcdf_member_count
        ),
        "file_rows": len(
            result.file_manifest
        ),
        "frame_rows": len(
            result.frame_manifest
        ),
        "schema_variants": len(
            result.schema_summary
        ),
        "build_errors": len(result.errors),
        "event_split_overlaps": len(
            validation.event_split_overlap
        ),
        "episode_split_overlaps": len(
            validation.episode_split_overlap
        ),
        "positive_frames": int(
            result.frame_manifest[
                "frame_label"
            ].sum()
        ),
        "total_frames": len(
            result.frame_manifest
        ),
        "terminal_marker": marker_name,
        "output_directory": str(
            output_directory
        ),
    }

    print()

    if status == "valid":
        print(
            f"PASS: {year} raw manifest is valid "
            "and was written"
        )
    else:
        print(
            f"AUDIT: {year} raw manifest was written "
            "with required validation failures"
        )

    for artifact_name, artifact_path in (
        artifacts.items()
    ):
        print(
            f"- {artifact_name}: "
            f"{artifact_path}"
        )

    return result, validation, summary


## Run the configured years


In [ ]:
annual_results = {}
annual_validations = {}
annual_summaries = []

for year in YEARS_TO_BUILD:
    result, validation, summary = (
        build_and_write_year(year)
    )

    annual_results[year] = result
    annual_validations[year] = validation
    annual_summaries.append(summary)

annual_summary_df = pd.DataFrame(
    annual_summaries
)

display(annual_summary_df)

invalid_years = (
    annual_summary_df.loc[
        annual_summary_df["status"]
        == "invalid",
        "year",
    ]
    .astype(int)
    .tolist()
)

if invalid_years:
    print(
        "AUDIT COMPLETE: raw manifests were "
        "preserved with required validation "
        f"failures for years {invalid_years}"
    )
else:
    print(
        "PASS: all configured annual raw "
        "manifests are valid"
    )
